In [22]:
import os
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11, 'figure.autolayout': True})

# ==========================================
# 1. ENVIRONMENT & DATA LOADING (POLARS)
# ==========================================
# Get current working directory (.\\notebooks or similar)
cwd = os.getcwd()

# Get parent folder to access (.\\Data)
parent = os.path.dirname(cwd)

# Get file path to load into environment
file_path_combined_data = os.path.join(parent, "Data", "combined_data.parquet")

print(f"Loading dataset from: {file_path_combined_data}")
df = pl.read_parquet(file_path_combined_data)
df.describe()

Loading dataset from: c:\Users\Marty\Documents\School\Merrimack College\8) DSE6311 Data Science Capstone\team_omega\Data\combined_data.parquet


statistic,ID,From,Date,Hour,Participation_Phase,Which_Phases_Participate,Demand_kWh,Hourly_cost_NOK,Hourly_cost_USD,Price_signal,Price_profile,Price_level,Experiment_price_NOK_kWh,NOK_USD_Rate,Experiment_price_USD_kWh,Temperature,Temperature24,Temperature48,Temperature72,Participation_Experiment,Recruitment_organisation,Municipality,Region,Survey1_answered,Survey2_answered,Survey3_answered,Control_Price_Phase1,Control_Price_Phase2,Group_Phase1,Group_Phase2,Participation_status_Phase1,Participation_status_Phase2
str,str,str,str,f64,str,str,f64,f64,f64,str,str,str,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str
"""count""","""12180000""","""12180000""","""12180000""",1.218e7,"""12180000""","""12180000""",1.218e7,1.46076e6,1.46076e6,"""1460760""","""1460760""","""1460760""",1.46076e6,1.05e7,1.46076e6,1.218e7,1.218e7,1.218e7,1.218e7,"""12180000""","""12180000""","""12180000""","""12180000""","""12180000""","""12180000""","""12180000""","""12180000""","""12180000""","""12180000""","""12180000""","""12180000""","""12180000"""
"""null_count""","""0""","""0""","""0""",0.0,"""0""","""0""",0.0,1.071924e7,1.071924e7,"""10719240""","""10719240""","""10719240""",1.071924e7,1.68e6,1.071924e7,0.0,0.0,0.0,0.0,"""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0"""
"""mean""",null,"""2021-01-27 23:29:59.999000+00:…","""2021-01-27 12:00:00""",12.5,null,null,2.564245,10.194558,1.195192,null,null,null,3.545642,0.117221,0.415601,-0.739941,-0.748828,-0.758594,-0.771649,null,null,null,null,null,null,null,null,null,null,null,null,null
"""std""",null,null,null,6.922187,null,null,1.99542,17.842131,2.09136,null,null,null,4.390327,0.001127,0.514567,5.124421,4.8099,4.67145,4.563512,null,null,null,null,null,null,null,null,null,null,null,null,null
"""min""","""Exp_1""","""2020-12-01 00:00:00+00:00""","""2020-12-01""",1.0,"""Phase_2""","""Both""",0.0,0.0,0.0,"""A_10""","""A""","""10""",0.0,0.1145,0.0,-21.8,-18.1,-17.3,-17.0,"""Yes""","""Electricity supplier 1""","""Asker""","""Bergen""","""NA""","""NA""","""No""","""Control group""","""Control group""","""Control""","""Ber_1""","""NA""","""No_information"""
"""25%""",null,"""2020-12-30 00:00:00+00:00""","""2020-12-30""",7.0,null,null,1.043,1.3144,0.154054,null,null,null,1.0,0.1163,0.1166,-4.3,-4.5,-4.5,-4.4,null,null,null,null,null,null,null,null,null,null,null,null,null
"""50%""",null,"""2021-01-28 00:00:00+00:00""","""2021-01-28""",13.0,null,null,2.151,3.4947,0.409962,null,null,null,1.1,0.1173,0.12793,0.0,0.3,0.4,0.4,null,null,null,null,null,null,null,null,null,null,null,null,null
"""75%""",null,"""2021-02-25 23:00:00+00:00""","""2021-02-25""",18.0,null,null,3.608,9.6228,1.128014,null,null,null,5.0,0.1181,0.587504,2.9,2.7,2.6,2.6,null,null,null,null,null,null,null,null,null,null,null,null,null
"""max""","""Exp_99""","""2021-03-26 23:00:00+00:00""","""2021-03-26""",24.0,"""Phase_2""","""Phase_2""",22.62,439.2855,51.703903,"""P_5""","""P0""","""5_10""",30.15,0.1194,3.560715,12.8,10.6,9.7,9.1,"""Yes""","""Transmission system operator""","""Trondheim""","""Trondheim""","""Yes""","""Yes""","""Yes""","""Price group""","""Price group""","""NA""","""Trond_Control""","""OK""","""OK"""


In [21]:
single_col = df.select(
    pl.col("Experiment_price_NOK_kWh").filter(pl.col("Experiment_price_NOK_kWh") != 0).min()
)
print(single_col)

shape: (1, 1)
┌──────────────────────────┐
│ Experiment_price_NOK_kWh │
│ ---                      │
│ f64                      │
╞══════════════════════════╡
│ 0.95                     │
└──────────────────────────┘


In [11]:
# ==========================================
# 2. FEATURE CLEANING & COLLINEARITY RESOLUTION
# ==========================================
# Drop 'Price_signal' to eliminate string concatenation redundancy & collinearity with Profile/Level.
# Keep 'Price_profile' (categorical shape) and 'Price_level' (numeric/ordinal magnitude).
if "Price_signal" in df.columns:
    df = df.drop("Price_signal")
    print("Dropped 'Price_signal' to prevent multicollinearity and parsing overhead.")

# Filter to Phase 2 for modeling and analysis (excluding Phase 1 COVID/selection bias)
df_phase2 = df.filter(
    (pl.col("Participation_Phase") == "Phase_2") & 
    (pl.col("Participation_status_Phase2") == "OK")
)

# df_phase2 = df[
#     (df["Participation_Phase"] == "Phase_2") & 
#     (df["Participation_status_Phase2"] == "OK")
# ].copy()

print(f"Loaded Phase 2 active rows: {df_phase2.height:,}")

Loaded Phase 2 active rows: 11,996,256


In [12]:
# ==========================================
# 3. TARGET ENGINEERING: FLEXIBILITY PROPENSITY INDEX (FPI)
# ==========================================
print("Engineering Flexibility Propensity Index (FPI)...")

# Define peak hours (e.g., hours 17-20 / 5 PM to 9 PM when price signals are active)
# Step A: Estimate household baseline consumption on non-experiment weekdays by hour & temperature bin
df_baselines = (
    df_phase2
    .filter(pl.col("Experiment_price_NOK_kWh").is_null())
    .with_columns(
        Temp_Bin = (pl.col("Temperature24") / 5).round(0) * 5 # Bin temperature by 5°C increments
    )
    .group_by(["ID", "Hour", "Temp_Bin"])
    .agg(
        Baseline_kWh = pl.col("Demand_kWh").mean()
    )
)

# Step B: Join baseline back to experiment days and compute FPI
df_experiment_days = (
    df_phase2
    .filter(pl.col("Experiment_price_NOK_kWh").is_not_null())
    .with_columns(
        Temp_Bin = (pl.col("Temperature24") / 5).round(0) * 5
    )
    .join(df_baselines, on=["ID", "Hour", "Temp_Bin"], how="left")
    # Fallback fill if exact temp bin baseline is missing
    .with_columns(
        Baseline_kWh = pl.col("Baseline_kWh").fill_null(pl.col("Demand_kWh").mean())
    )
)

# Step C: Aggregate to Household x Experiment-Day Level FPI
df_fpi_daily = (
    df_experiment_days
    .group_by(["ID", "Date", "Region", "Price_profile", "Price_level"])
    .agg(
        Actual_Peak_Demand = pl.col("Demand_kWh").sum(),
        Base_Peak_Demand = pl.col("Baseline_kWh").sum(),
        Mean_Temp = pl.col("Temperature").mean()
    )
    .with_columns(
        # FPI formula: Percentage reduction relative to baseline (winsorized)
        Flexibility_Propensity_Index = (
            (pl.col("Base_Peak_Demand") - pl.col("Actual_Peak_Demand")) / (pl.col("Base_Peak_Demand") + 1e-6) * 100
        ).clip(-50.0, 50.0),
        High_Flexibility_Flag = (
            ((pl.col("Base_Peak_Demand") - pl.col("Actual_Peak_Demand")) / (pl.col("Base_Peak_Demand") + 1e-6) * 100) >= 5.0
        ).cast(pl.Int32)
    )
)

print(f"Computed FPI across {df_fpi_daily.height:,} household-experiment days.")

Engineering Flexibility Propensity Index (FPI)...
Computed FPI across 59,677 household-experiment days.


In [13]:
# ==========================================
# 4. APPENDIX B: GENERATING TABLES & FIGURES
# ==========================================

# --- TABLE B1: Dataset Dimensionality & Observation Breakdown ---
table_b1 = (
    df
    .group_by(["Participation_Phase", "Participation_Experiment", "Control_Price_Phase2"])
    .agg(
        Total_Rows = pl.len(),
        Unique_Households = pl.col("ID").n_unique()
    )
    .sort(["Participation_Phase", "Participation_Experiment"])
)

print("\n=== Table B1: Dataset Dimensionality & Observation Breakdown ===")
print(table_b1)


=== Table B1: Dataset Dimensionality & Observation Breakdown ===
shape: (2, 5)
┌─────────────────────┬─────────────────────┬─────────────────────┬────────────┬───────────────────┐
│ Participation_Phase ┆ Participation_Exper ┆ Control_Price_Phase ┆ Total_Rows ┆ Unique_Households │
│ ---                 ┆ iment               ┆ 2                   ┆ ---        ┆ ---               │
│ str                 ┆ ---                 ┆ ---                 ┆ u32        ┆ u32               │
│                     ┆ str                 ┆ str                 ┆            ┆                   │
╞═════════════════════╪═════════════════════╪═════════════════════╪════════════╪═══════════════════╡
│ Phase_2             ┆ Yes                 ┆ Control group       ┆ 3541248    ┆ 1272              │
│ Phase_2             ┆ Yes                 ┆ Price group         ┆ 8638752    ┆ 3103              │
└─────────────────────┴─────────────────────┴─────────────────────┴────────────┴───────────────────┘


In [14]:
# --- FIGURE B1: Distribution of Hourly Demand & Outdoor Temperature by Region ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Convert sample to pandas for plotting with seaborn
pdf_sample = df_phase2.select(["Region", "Demand_kWh", "Temperature24"]).sample(n=50000, seed=42).to_pandas()

sns.boxplot(data=pdf_sample, x="Region", y="Demand_kWh", ax=axes[0], palette="Blues")
axes[0].set_title("Hourly Electricity Demand by Region (Phase 2)")
axes[0].set_xlabel("Region")
axes[0].set_ylabel("Demand (kWh)")
axes[0].tick_params(axis='x', rotation=30)

sns.boxplot(data=pdf_sample, x="Region", y="Temperature24", ax=axes[1], palette="coolwarm")
axes[1].set_title("24-Hour Rolling Temperature by Region (°C)")
axes[1].set_xlabel("Region")
axes[1].set_ylabel("Temperature (°C)")
axes[1].tick_params(axis='x', rotation=30)

fig_path_b1 = os.path.join(parent, "Data", "Figure_B1_Demand_Temperature_Distribution.png")
plt.savefig(fig_path_b1, dpi=300, bbox_inches="tight")
print(f"\nSaved Figure B1 to: {fig_path_b1}")
plt.close()

C:\Users\Marty\AppData\Local\Temp\ipykernel_42012\4282673827.py:7: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=pdf_sample, x="Region", y="Demand_kWh", ax=axes[0], palette="Blues")
C:\Users\Marty\AppData\Local\Temp\ipykernel_42012\4282673827.py:13: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=pdf_sample, x="Region", y="Temperature24", ax=axes[1], palette="coolwarm")



Saved Figure B1 to: c:\Users\Marty\Documents\School\Merrimack College\8) DSE6311 Data Science Capstone\team_omega\Data\Figure_B1_Demand_Temperature_Distribution.png


In [ ]:
# ==========================================
# 3. FIGURE B2 FIX: DIURNAL LOAD PROFILE
# ==========================================
df_diurnal = (
    df_phase2.assign(Is_Experiment=df_phase2["Experiment_price_NOK_kWh"].notna())
    .groupby(["Hour", "Is_Experiment"])["Demand_kWh"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
palette_dict = {False: "#1b9e77", True: "#d95f02"}

sns.lineplot(
    data=df_diurnal, 
    x="Hour", 
    y="Demand_kWh", 
    hue="Is_Experiment", 
    palette=palette_dict,
    marker="o", 
    linewidth=2.5,
    ax=ax,
    legend=False # Disable automatic buggy legend
)

ax.set_title("Mean Diurnal Load Profile: Experiment Days vs. Non-Experiment Days")
ax.set_xlabel("Hour of Day (1–24)")
ax.set_ylabel("Mean Hourly Electricity Demand (kWh)")
ax.set_xticks(range(1, 25))

# Explicitly create correct legend handles with matching colors
legend_handles = [
    mlines.Line2D([], [], color="#1b9e77", marker='o', linewidth=2.5, label="Non-Experiment Baseline"),
    mlines.Line2D([], [], color="#d95f02", marker='o', linewidth=2.5, label="Active Price Experiment Day")
]
ax.legend(handles=legend_handles, title="Condition", loc="upper left")


# Explicit legend handles with distinct colors to avoid matching line colors directly
# legend_handles = [
#     mlines.Line2D([], [], color="#7570b3", marker='s', linewidth=2.5, label="Non-Experiment Baseline"),
#     mlines.Line2D([], [], color="#e7298a", marker='^', linewidth=2.5, label="Active Price Experiment Day")
# ]
# ax.legend(handles=legend_handles, title="Experimental Condition", loc="upper left")

fig_path_b2 = os.path.join(parent, "Data", "Figure_B2_Diurnal_Load_Profiles.png")
plt.savefig(fig_path_b2, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved fixed Figure B2 to: {fig_path_b2}")

AttributeError: 'DataFrame' object has no attribute 'assign'